In [1]:
#!/usr/bin/env python3
"""
HK Address Parser – Inference for LLM (Qwen2.5-3B-Instruct + LoRA)
Hardened for Tesla P40 / Pascal (sm_61)
"""
import os
import re
import subprocess
import sys

# ---- MUST run before importing torch / transformers / bitsandbytes ----
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

def _lock_emptiest_gpu():
    """Expose one GPU as cuda:0. bnb/transformers are unsafe on cuda:1+."""
    if "CUDA_VISIBLE_DEVICES" in os.environ:
        return
    try:
        raw = subprocess.check_output(
            ["nvidia-smi",
             "--query-gpu=index,memory.free,utilization.gpu",
             "--format=csv,nounits,noheader"],
            encoding="utf-8",
        )
        best_id, max_free, fallback = None, -1, "0"
        for line in raw.strip().splitlines():
            gid, free, util = [p.strip() for p in line.split(",")]
            gid, free, util = int(gid), int(free), int(util)
            fallback = str(gid)
            if free > max_free and util < 30:
                max_free, best_id = free, str(gid)
        os.environ["CUDA_VISIBLE_DEVICES"] = best_id if best_id is not None else fallback
    except Exception:
        pass

_lock_emptiest_gpu()

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

try:
    from peft import PeftModel
except ImportError:
    PeftModel = None

# Kill fused SDPA — Pascal segfault magnet
if torch.cuda.is_available():
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)


class HKAddressParserLLM:
    def __init__(self, base_model_path, lora_path, conf_threshold=0.50, max_new_tokens=128):
        self.base_model_path = base_model_path
        self.lora_path = lora_path
        self.conf_threshold = conf_threshold
        self.max_new_tokens = max_new_tokens

        # After CUDA_VISIBLE_DEVICES, the chosen card is always cuda:0
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.use_quant = False

        print(f"DEBUG: CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')}")
        print(f"DEBUG: Using Device → {self.device}")

        if torch.cuda.is_available():
            cap = torch.cuda.get_device_capability(0)
            name = torch.cuda.get_device_name(0)
            print(f"DEBUG: GPU → {name}  capability={cap}")
            # Pascal = major < 7 (P40 is 6.1). Skip bitsandbytes entirely.
            self.is_pascal = cap[0] < 7
            torch.cuda.empty_cache()
            try:
                free, total = torch.cuda.mem_get_info(0)
                print(f"DEBUG: Free VRAM: {free / 1024**3:.1f} GB / {total / 1024**3:.1f} GB")
            except Exception:
                pass
        else:
            self.is_pascal = True

        print("📦 Loading Tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(
            base_model_path,
            trust_remote_code=True,
            padding_side="left",
        )
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token_id = self.tokenizer.eos_token_id

        print(f"📦 Loading Base Model from {base_model_path}...")
        load_kwargs = dict(
            trust_remote_code=True,
            attn_implementation="eager",
            device_map={"": 0} if torch.cuda.is_available() else None,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            low_cpu_mem_usage=True,
        )

        # 3B fp16 fits easily in 24 GB. 4-bit on P40 is what segfaults.
        if torch.cuda.is_available() and not self.is_pascal:
            try:
                from transformers import BitsAndBytesConfig
                load_kwargs["quantization_config"] = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_quant_type="nf4",
                    bnb_4bit_use_double_quant=True,
                )
                self.use_quant = True
                print("📦 4-bit quantization enabled (Turing+ GPU)")
            except Exception as e:
                print(f"⚠️ bitsandbytes unavailable ({e}); using fp16")
        else:
            print("📦 Pascal/P40 detected (or no CUDA) → fp16, no bitsandbytes")

        base_model = AutoModelForCausalLM.from_pretrained(base_model_path, **load_kwargs)

        if os.path.exists(lora_path) and PeftModel is not None:
            print(f"📦 Loading LoRA adapters from {lora_path}...")
            self.model = PeftModel.from_pretrained(base_model, lora_path, is_trainable=False)
        else:
            if not os.path.exists(lora_path):
                print(f"⚠️ LoRA path {lora_path} not found. Using base model only.")
            self.model = base_model

        self.model.eval()
        if hasattr(self.model, "config"):
            self.model.config.use_cache = True
        print("✅ Fine-tuned model ready!")

    def _build_prompts(self, addresses):
        system_prompt = (
            "You are an expert Hong Kong Address Formatter. "
            "Split the provided address into exactly two lines based on HK formatting rules. "
            "Output strictly in this format:\nLine 1: <macro elements>\nLine 2: <micro elements>"
        )
        prompts = []
        for addr in addresses:
            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": addr},
            ]
            prompts.append(
                self.tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )
            )
        return prompts

    def _parse_llm_output(self, generated_text):
        line1, line2 = "", ""
        l1_match = re.search(r"Line\s*1:\s*(.+)", generated_text, re.IGNORECASE)
        l2_match = re.search(r"Line\s*2:\s*(.+)", generated_text, re.IGNORECASE)
        if l1_match:
            line1 = l1_match.group(1).strip()
        if l2_match:
            line2 = l2_match.group(1).strip()
        return line1, line2

    def parse_batch(self, address_pairs, batch_size=1):
        all_results = []
        for i in range(0, len(address_pairs), batch_size):
            batch_pairs = address_pairs[i:i + batch_size]
            full_addresses = []
            for p1, p2 in batch_pairs:
                parts = [p.strip() for p in (p1, p2) if p and p.strip()]
                if not parts:
                    full_addresses.append("")
                elif len(parts) == 1:
                    full_addresses.append(parts[0])
                else:
                    combined = parts[0] + parts[1]
                    is_chinese = any("\u4e00" <= c <= "\u9fff" for c in combined)
                    if is_chinese:
                        full_addresses.append(" ".join(parts))
                    elif parts[0].endswith(","):
                        full_addresses.append(f"{parts[0]} {parts[1]}")
                    else:
                        full_addresses.append(f"{parts[0]}, {parts[1]}")

            valid_indices = [idx for idx, addr in enumerate(full_addresses) if addr]
            valid_addresses = [full_addresses[idx] for idx in valid_indices]
            decoded_texts = []
            split_confs = []

            if valid_addresses:
                prompts = self._build_prompts(valid_addresses)
                inputs = self.tokenizer(
                    prompts,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=2048,
                )
                inputs = {k: v.to(self.device) for k, v in inputs.items()}

                # No output_scores on Pascal — that path has extra kernels and
                # cannot be caught with try/except if it segfaults.
                with torch.inference_mode():
                    sequences = self.model.generate(
                        **inputs,
                        max_new_tokens=self.max_new_tokens,
                        do_sample=False,
                        use_cache=True,
                        pad_token_id=self.tokenizer.pad_token_id,
                        eos_token_id=self.tokenizer.eos_token_id,
                    )

                gen_tokens = sequences[:, inputs["input_ids"].shape[1]:]
                decoded_texts = self.tokenizer.batch_decode(
                    gen_tokens, skip_special_tokens=True
                )
                split_confs = [0.75] * len(decoded_texts)

            valid_idx_ptr = 0
            for idx, address_str in enumerate(full_addresses):
                if not address_str:
                    all_results.append(("", "", "", {}, {}, 0.0, [], "EMPTY_INPUT"))
                    continue

                raw_output = decoded_texts[valid_idx_ptr]
                split_conf = split_confs[valid_idx_ptr]
                valid_idx_ptr += 1
                line1, line2 = self._parse_llm_output(raw_output)

                def normalize_for_check(text):
                    if not text:
                        return ""
                    return re.sub(r"[\s,/\\\-;\.，。、；]+", "", str(text).lower())

                is_reversed = normalize_for_check(address_str) != normalize_for_check(line1 + line2)
                warnings, alerts = [], []
                if not line1 and not line2:
                    alerts.append("LLM failed to format Line 1 and Line 2 properly")
                if is_reversed:
                    alerts.append("reversed order after parsing")
                if split_conf < self.conf_threshold:
                    warnings.append(
                        f"low splitting confidence ({split_conf:.4f} < {self.conf_threshold})"
                    )

                status = ""
                if alerts:
                    status += "[ALERT]: " + "; ".join(alerts)
                if warnings:
                    status += "[WARNING]: " + "; ".join(warnings)
                if not status:
                    status = "[SUCCESS]"

                all_results.append((
                    address_str, line1, line2,
                    {}, {}, split_conf, [], status
                ))
        return all_results

    def parse(self, address_part1, address_part2=""):
        return self.parse_batch([(address_part1, address_part2)], batch_size=1)[0]


if __name__ == "__main__":
    BASE_MODEL = "models/Qwen2.5-3B-Instruct"
    LORA_DIR = "Qwen2.5-3B-Instruct_Address_Formatter"

    print("Initializing LLM Parser...")
    parser = HKAddressParserLLM(base_model_path=BASE_MODEL, lora_path=LORA_DIR)

    test_cases = [
        ("九龍觀塘區雲漢街61號南寧大樓地庫01舖", ""),
        (
            "ROOM 2107, 42/F, WINNING HEIGHTS, 277 CASTLE PEAK ROAD, "
            "TSUEN WAN, TSUEN WAN, Ma Wan, TSUEN WAN DISTRICT, New Territories",
            "",
        ),
    ]

    print("\nRunning tests:\n" + "=" * 70)
    results = parser.parse_batch(test_cases, batch_size=1)
    for i, (a1, a2) in enumerate(test_cases):
        addr, l1, l2, tags, confs, overall, used_keys, status = results[i]
        print(f"Original : {addr}")
        print(f"Line 1 : {l1}")
        print(f"Line 2 : {l2}")
        print(f"Tags : {tags} (Empty for LLM)")
        print(f"Conf : {confs} (Empty for LLM)")
        print(f"Logic keys: {used_keys} | Split conf: {overall:.4f}")
        print(f"status : {status}")
        print("-" * 70)

Initializing LLM Parser...
DEBUG: CUDA_VISIBLE_DEVICES=1
DEBUG: Using Device → cuda:0
DEBUG: GPU → Tesla P40  capability=(6, 1)
DEBUG: Free VRAM: 22.2 GB / 22.4 GB
📦 Loading Tokenizer...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


📦 Loading Base Model from models/Qwen2.5-3B-Instruct...
📦 Pascal/P40 detected (or no CUDA) → fp16, no bitsandbytes


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

📦 Loading LoRA adapters from Qwen2.5-3B-Instruct_Address_Formatter...
✅ Fine-tuned model ready!

Running tests:
Original : 九龍觀塘區雲漢街61號南寧大樓地庫01舖
Line 1 : 九龍觀塘區雲漢街61號
Line 2 : 南寧大樓地庫01舖
Tags : {} (Empty for LLM)
Conf : {} (Empty for LLM)
Logic keys: [] | Split conf: 0.7500
status : [SUCCESS]
----------------------------------------------------------------------
Original : ROOM 2107, 42/F, WINNING HEIGHTS, 277 CASTLE PEAK ROAD, TSUEN WAN, TSUEN WAN, Ma Wan, TSUEN WAN DISTRICT, New Territories
Line 1 : ROOM 2107, 42/F, WINNING HEIGHTS
Line 2 : 277 CASTLE PEAK ROAD, TSUEN WAN, TSUEN WAN DISTRICT, New Territories
Tags : {} (Empty for LLM)
Conf : {} (Empty for LLM)
Logic keys: [] | Split conf: 0.7500
status : [ALERT]: reversed order after parsing
----------------------------------------------------------------------


In [3]:
!python --version

Python 3.9.25
